Four tables are displayed and exported: three detailed tables for tracing feasibility, importance, and barrier results, plus one combined table containing only overall summaries. The detailed tables retain category/item rows and their effect estimates and confidence intervals.

- Feasibility reports the percentage rating the single question **very feasible (4)**. The remaining responses are **feasible (3)**; no agreement coefficient is reported for this single rated object.
- Importance reports ordinal Krippendorff's alpha across the three shared technologies, with 95% expert-bootstrap confidence intervals.
- Barriers reports nominal Krippendorff's alpha across the 11 barriers, with the same confidence intervals.
- Each row includes the existing overall group/session comparison and its Holm-adjusted p-value (three comparisons). These tests compare response distributions, not the agreement coefficients.

Percentages contain no count fractions. Alpha coefficients remain on their original scale. The combined LaTeX table is saved to `latex-tables/agreement_overall_summary.tex` beside this notebook; it requires `booktabs` and `tabularx`.

Detailed LaTeX exports are `agreement_feasibility.tex`, `agreement_importance.tex`, and `agreement_barriers.tex` in the same `latex-tables` folder. All proportion entries use percentages without count fractions.


In [1]:
# Cell 1: Load and validate the data
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import krippendorff

from scipy.stats import fisher_exact
from IPython.display import display

ROOT = Path(
    "/home/ngirwa/TiiS/second-publication/"
    "2-current-analysis/rq1-rerun/raw-responses"
)

N_BOOT = 10_000
N_PERM = 49_999
rng = np.random.default_rng(20261119)

SHARED_SOLUTIONS = [
    "Smart and sustainable DC cables",
    "DC connectors",
    "LVAC-LVDC interlink converter",
]

feas = pd.read_csv(ROOT / "feasibility_humanOnly_rq1.csv")
imp = pd.read_csv(ROOT / "importance_humanOnly_rq1.csv")
bar = pd.read_csv(ROOT / "barriers_human_final.csv")

# Standardize identifiers in memory.
feas = feas.rename(
    columns={"variant_id": "expert", "model": "session"}
)
imp = imp.rename(
    columns={
        "respondent_id": "expert",
        "variant_id": "session",
        "rating_numeric": "rating",
    }
)

bar[["expert", "session"]] = bar["row_id"].str.rsplit(
    "_", n=1, expand=True
)

for df in (feas, imp, bar):
    for col in ["expert", "session"]:
        df[col] = df[col].astype(str).str.strip()

feas["rating"] = pd.to_numeric(feas["rating"], errors="raise")
imp["rating"] = pd.to_numeric(imp["rating"], errors="raise")
bar["barrier_id"] = pd.to_numeric(
    bar["barrier_id"], errors="raise"
)

assert not feas.duplicated(["session", "expert"]).any()
assert not imp.duplicated(
    ["session", "expert", "technology"]
).any()
assert not bar.duplicated(
    ["session", "expert", "barrier_id"]
).any()

assert feas["session"].isin(["s1", "s2"]).all()
assert imp["session"].isin(["s1", "s2"]).all()
assert bar["session"].isin(["s1", "s2"]).all()

assert feas["rating"].isin([3, 4]).all(), (
    "Feasibility now contains categories other than 3/4; "
    "revise the Fisher table before proceeding."
)
assert imp["rating"].isin([1, 2, 3, 4]).all()
assert bar["barrier_id"].isin(range(1, 12)).all()

roster = {
    session: sorted(
        feas.loc[feas["session"] == session, "expert"]
    )
    for session in ["s1", "s2"]
}
assert len(roster["s1"]) == 13
assert len(roster["s2"]) == 11

for session in ["s1", "s2"]:
    expected = set(roster[session])
    assert set(
        imp.loc[imp["session"] == session, "expert"]
    ) == expected
    assert set(
        bar.loc[bar["session"] == session, "expert"]
    ) == expected

# Importance: only the three genuinely shared technologies.
imp_wide = imp.loc[
    imp["technology"].isin(SHARED_SOLUTIONS)
].pivot(
    index=["session", "expert"],
    columns="technology",
    values="rating",
).reindex(columns=SHARED_SOLUTIONS)

assert not imp_wide.isna().any().any()

# Barriers: 0 means not selected by a responding expert.
bar_wide = (
    bar.assign(selected=1)
    .pivot(
        index=["session", "expert"],
        columns="barrier_id",
        values="selected",
    )
    .reindex(columns=range(1, 12))
    .fillna(0)
)

def group_arrays(wide):
    return tuple(
        wide.xs(session, level="session")
        .reindex(roster[session])
        .to_numpy(dtype=float)
        for session in ["s1", "s2"]
    )

I1, I2 = group_arrays(imp_wide)
B1, B2 = group_arrays(bar_wide)

F1, F2 = tuple(
    feas.loc[feas["session"] == session]
    .set_index("expert")
    .reindex(roster[session])["rating"]
    .to_numpy(dtype=float)
    for session in ["s1", "s2"]
)

assert I1.shape == (13, 3)
assert I2.shape == (11, 3)
assert B1.shape == (13, 11)
assert B2.shape == (11, 11)

print(
    "Validated: 13 consortium experts, 11 industry experts; "
    "3 shared technologies and 11 barriers."
)

Validated: 13 consortium experts, 11 industry experts; 3 shared technologies and 11 barriers.


In [2]:
#Cell 2 — Agreement, confidence intervals, and group-comparison functions
def interval(values):
    values = np.asarray(values, dtype=float)
    valid = np.isfinite(values)

    if valid.mean() < 0.95:
        warnings.warn(
            "More than 5% of bootstrap estimates are undefined. "
            "Interpret this confidence interval cautiously."
        )
    if not valid.any():
        return np.nan, np.nan

    return tuple(np.quantile(values[valid], [0.025, 0.975]))


def estimate_ci(estimate, bounds, scale=1):
    if not np.isfinite(estimate):
        return "Undefined"
    lo, hi = bounds
    return (
        f"{estimate * scale:.3f} "
        f"[{lo * scale:.3f}, {hi * scale:.3f}]"
    )


def alpha_value(matrix, level):
    """Rows = experts; columns = common rated objects."""
    if np.unique(matrix).size < 2:
        return np.nan

    return float(
        krippendorff.alpha(
            reliability_data=matrix,
            level_of_measurement=level,
            value_domain=(
                np.array([1, 2, 3, 4])
                if level == "ordinal"
                else np.array([0, 1])
            ),
        )
    )


def alpha_results(a, b, level):
    """
    Bootstrap experts, preserving their complete response profiles.
    Pooled resampling preserves the two group sizes.
    """
    point = [
        alpha_value(a, level),
        alpha_value(b, level),
        alpha_value(np.vstack([a, b]), level),
    ]

    boot = np.empty((N_BOOT, 3))

    for k in range(N_BOOT):
        aa = a[rng.integers(0, len(a), size=len(a))]
        bb = b[rng.integers(0, len(b), size=len(b))]

        boot[k] = [
            alpha_value(aa, level),
            alpha_value(bb, level),
            alpha_value(np.vstack([aa, bb]), level),
        ]

    return [
        estimate_ci(point[j], interval(boot[:, j]))
        for j in range(3)
    ]


def cliffs_delta(a, b):
    """Positive: industry (b) tends to rate higher."""
    return np.sign(
        b[:, None, :] - a[None, :, :]
    ).mean(axis=(0, 1))


def effect_results(a, b, kind):
    """
    kind='ordinal': Cliff's delta per solution.
    kind='binary': industry minus consortium proportion.
    """
    if kind == "ordinal":
        point = cliffs_delta(a, b)
    else:
        point = b.mean(axis=0) - a.mean(axis=0)

    boot = np.empty((N_BOOT, a.shape[1]))

    for k in range(N_BOOT):
        aa = a[rng.integers(0, len(a), size=len(a))]
        bb = b[rng.integers(0, len(b), size=len(b))]

        boot[k] = (
            cliffs_delta(aa, bb)
            if kind == "ordinal"
            else bb.mean(axis=0) - aa.mean(axis=0)
        )

    return point, [
        interval(boot[:, j])
        for j in range(a.shape[1])
    ]


def permutation_profile_test(a, b):
    """
    One omnibus comparison of the complete response profiles.

    Statistic: sum of squared differences between group means
    of the supplied features. Whole experts are permuted.
    """
    z = np.vstack([a, b])
    n1 = len(a)

    observed = np.square(
        a.mean(axis=0) - b.mean(axis=0)
    ).sum()

    extreme = 0
    completed = 0

    while completed < N_PERM:
        batch = min(1000, N_PERM - completed)

        # Independent random permutations of expert indices.
        order = np.argsort(
            rng.random((batch, len(z))), axis=1
        )

        mean1 = z[order[:, :n1]].mean(axis=1)
        mean2 = z[order[:, n1:]].mean(axis=1)
        statistics = np.square(mean1 - mean2).sum(axis=1)

        extreme += np.count_nonzero(
            statistics >= observed - 1e-12
        )
        completed += batch

    return (extreme + 1) / (N_PERM + 1)


def ordinal_features(matrix):
    """
    Cumulative category indicators preserve ordinal ordering
    without assuming equal distances between rating categories.
    """
    return (
        matrix[:, :, None] <= np.array([1, 2, 3])
    ).reshape(len(matrix), -1).astype(float)


def holm_adjust(pvalues):
    pvalues = np.asarray(pvalues)
    order = np.argsort(pvalues)
    adjusted = np.empty(len(pvalues))

    adjusted[order] = np.minimum(
        1,
        np.maximum.accumulate(
            pvalues[order]
            * np.arange(len(pvalues), 0, -1)
        ),
    )
    return adjusted


def median_iqr(values):
    q1, median, q3 = np.quantile(
        values, [0.25, 0.50, 0.75]
    )
    return f"{median:g} [{q1:g}, {q3:g}]"


def percent(values):
    """Format a binary indicator as a percentage without count fractions."""
    return f"{100 * np.mean(values):.1f}%"

In [3]:
# Cell 3 — Run the evaluation and build detailed and overall tables
feasibility_counts = np.array([
    [(F1 == 3).sum(), (F1 == 4).sum()],
    [(F2 == 3).sum(), (F2 == 4).sum()],
])

p_feas = fisher_exact(
    feasibility_counts, alternative="two-sided"
).pvalue

# Importance: compare cumulative rating distributions jointly
# across the three shared solutions.
p_imp = permutation_profile_test(
    ordinal_features(I1),
    ordinal_features(I2),
)

# Barriers: compare the eleven selection probabilities jointly.
p_bar = permutation_profile_test(B1, B2)

p_adjusted = holm_adjust([p_feas, p_imp, p_bar])


# ---------- Agreement ----------

print("Computing agreement confidence intervals...")

imp_alpha = alpha_results(I1, I2, level="ordinal")
bar_alpha = alpha_results(B1, B2, level="nominal")


# ---------- Effects and confidence intervals ----------

feas_effect, feas_bounds = effect_results(
    (F1 == 4).astype(float)[:, None],
    (F2 == 4).astype(float)[:, None],
    kind="binary",
)

imp_effect, imp_bounds = effect_results(
    I1, I2, kind="ordinal"
)

bar_effect, bar_bounds = effect_results(
    B1, B2, kind="binary"
)


# ---------- Table 1: Feasibility ----------

table_feasibility = pd.DataFrame([
    {
        "Measure": "Feasible (rating 3)",
        "Consortium": percent(F1 == 3),
        "Industry": percent(F2 == 3),
        "Industry − consortium, pp [95% CI]": estimate_ci(
            -feas_effect[0],
            (-feas_bounds[0][1], -feas_bounds[0][0]),
            scale=100,
        ),
        "Fisher p": np.nan,
        "Holm-adjusted p": np.nan,
    },
    {
        "Measure": "Very feasible (rating 4)",
        "Consortium": percent(F1 == 4),
        "Industry": percent(F2 == 4),
        "Industry − consortium, pp [95% CI]": estimate_ci(
            feas_effect[0], feas_bounds[0], scale=100
        ),
        "Fisher p": np.nan,
        "Holm-adjusted p": np.nan,
    },
    {
        "Measure": "Overall group/session comparison",
        "Consortium": "—",
        "Industry": "—",
        "Industry − consortium, pp [95% CI]": "—",
        "Fisher p": p_feas,
        "Holm-adjusted p": p_adjusted[0],
    },
])


# ---------- Table 2: Importance ----------

importance_rows = [{
    "Measure": "Ordinal alpha across shared solutions",
    "Consortium": imp_alpha[0],
    "Industry": imp_alpha[1],
    "Pooled alpha / Cliff's delta [95% CI]": imp_alpha[2],
    "Permutation p": np.nan,
    "Holm-adjusted p": np.nan,
}]

for j, solution in enumerate(SHARED_SOLUTIONS):
    importance_rows.append({
        "Measure": solution,
        "Consortium": median_iqr(I1[:, j]),
        "Industry": median_iqr(I2[:, j]),
        "Pooled alpha / Cliff's delta [95% CI]": estimate_ci(
            imp_effect[j], imp_bounds[j]
        ),
        "Permutation p": np.nan,
        "Holm-adjusted p": np.nan,
    })

importance_rows.append({
    "Measure": "Overall group/session comparison",
    "Consortium": "—",
    "Industry": "—",
    "Pooled alpha / Cliff's delta [95% CI]": "—",
    "Permutation p": p_imp,
    "Holm-adjusted p": p_adjusted[1],
})

table_importance = pd.DataFrame(importance_rows)


# ---------- Table 3: Barriers ----------

barrier_rows = [{
    "Measure": "Nominal alpha across 11 barriers",
    "Consortium": bar_alpha[0],
    "Industry": bar_alpha[1],
    "Pooled alpha / difference, pp [95% CI]": bar_alpha[2],
    "Permutation p": np.nan,
    "Holm-adjusted p": np.nan,
}]

for j in range(11):
    barrier_rows.append({
        "Measure": f"B{j + 1}",
        "Consortium": percent(B1[:, j]),
        "Industry": percent(B2[:, j]),
        "Pooled alpha / difference, pp [95% CI]": estimate_ci(
            bar_effect[j], bar_bounds[j], scale=100
        ),
        "Permutation p": np.nan,
        "Holm-adjusted p": np.nan,
    })

barrier_rows.append({
    "Measure": "Overall group/session comparison",
    "Consortium": "—",
    "Industry": "—",
    "Pooled alpha / difference, pp [95% CI]": "—",
    "Permutation p": p_bar,
    "Holm-adjusted p": p_adjusted[2],
})

table_barriers = pd.DataFrame(barrier_rows)

# ---------- Combined overall summary (no per-item rows) ----------
# The feasibility entries are percentages; importance/barrier entries are
# agreement coefficients. Keep the measures explicit in the row labels.
table_overall = pd.DataFrame([
    {
        "Measure": "Feasibility: very feasible (%)",
        "Consortium": percent(F1 == 4),
        "Industry": percent(F2 == 4),
        "Pooled alpha [95% CI]": "—",
        "Overall p": p_feas,
        "Holm-adjusted p": p_adjusted[0],
    },
    {
        "Measure": "Importance: ordinal α",
        "Consortium": imp_alpha[0],
        "Industry": imp_alpha[1],
        "Pooled alpha [95% CI]": imp_alpha[2],
        "Overall p": p_imp,
        "Holm-adjusted p": p_adjusted[1],
    },
    {
        "Measure": "Barriers: nominal α",
        "Consortium": bar_alpha[0],
        "Industry": bar_alpha[1],
        "Pooled alpha [95% CI]": bar_alpha[2],
        "Overall p": p_bar,
        "Holm-adjusted p": p_adjusted[2],
    },
])


Computing agreement confidence intervals...


In [4]:
from IPython.display import display, HTML

def show_table(title, table):
    formatted = table.copy()

    for column in formatted.columns:
        if column.endswith(" p"):
            formatted[column] = formatted[column].map(
                lambda value: (
                    "—" if pd.isna(value)
                    else f"{value:.4f}"
                )
            )

    print(title)
    display(HTML(
        formatted.to_html(
            index=False,
            na_rep="—",
            escape=True,
        )
    ))

show_table("Table 1. Feasibility (detailed)", table_feasibility)
show_table("Table 2. Importance (detailed)", table_importance)
show_table("Table 3. Barriers (detailed)", table_barriers)
show_table("Table 4. Overall expert summaries and group/session comparisons", table_overall)


Table 1. Feasibility (detailed)
Table 2. Importance (detailed)
Table 3. Barriers (detailed)
Table 4. Overall expert summaries and group/session comparisons


Measure,Consortium,Industry,"Industry − consortium, pp [95% CI]",Fisher p,Holm-adjusted p
Feasible (rating 3),84.6%,63.6%,"-20.979 [-55.944, 12.587]",—,—
Very feasible (rating 4),15.4%,36.4%,"20.979 [-12.587, 55.944]",—,—
Overall group/session comparison,—,—,—,0.3572,0.8182


Measure,Consortium,Industry,Pooled alpha / Cliff's delta [95% CI],Permutation p,Holm-adjusted p
Ordinal alpha across shared solutions,"-0.031 [-0.053, 0.181]","0.005 [-0.044, 0.264]","-0.002 [-0.027, 0.120]",—,—
Smart and sustainable DC cables,"4 [3, 4]","3 [3, 4]","-0.161 [-0.552, 0.252]",—,—
DC connectors,"4 [4, 4]","4 [2.5, 4]","-0.287 [-0.657, 0.105]",—,—
LVAC-LVDC interlink converter,"4 [3, 4]","4 [4, 4]","0.070 [-0.287, 0.448]",—,—
Overall group/session comparison,—,—,—,0.2727,0.8182


Measure,Consortium,Industry,"Pooled alpha / difference, pp [95% CI]",Permutation p,Holm-adjusted p
Nominal alpha across 11 barriers,"0.087 [0.006, 0.328]","0.045 [-0.012, 0.339]","0.070 [0.021, 0.221]",—,—
B1,30.8%,36.4%,"5.594 [-30.769, 43.357]",—,—
B2,15.4%,27.3%,"11.888 [-21.678, 45.455]",—,—
B3,23.1%,54.5%,"31.469 [-9.790, 66.434]",—,—
B4,53.8%,27.3%,"-26.573 [-61.538, 13.287]",—,—
B5,30.8%,36.4%,"5.594 [-30.769, 41.958]",—,—
B6,38.5%,54.5%,"16.084 [-25.175, 55.944]",—,—
B7,23.1%,18.2%,"-4.895 [-37.063, 28.671]",—,—
B8,53.8%,54.5%,"0.699 [-40.559, 41.958]",—,—
B9,76.9%,72.7%,"-4.196 [-39.161, 29.371]",—,—


Measure,Consortium,Industry,Pooled alpha [95% CI],Overall p,Holm-adjusted p
Feasibility: very feasible (%),15.4%,36.4%,—,0.3572,0.8182
Importance: ordinal α,"-0.031 [-0.053, 0.181]","0.005 [-0.044, 0.264]","-0.002 [-0.027, 0.120]",0.2727,0.8182
Barriers: nominal α,"0.087 [0.006, 0.328]","0.045 [-0.012, 0.339]","0.070 [0.021, 0.221]",0.5819,0.8182


In [5]:
from pathlib import Path
import pandas as pd

EXPORT_DIR = ROOT.parent / "experts-consistency" / "latex-tables"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)


def latex_escape(value):
    """Escape table text without requiring pandas Styler or jinja2."""
    if pd.isna(value):
        return "---"

    replacements = {
        "\\": r"\textbackslash{}",
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
        "−": r"\ensuremath{-}",
        "—": "---",
        "–": "--",
        "α": r"\ensuremath{\alpha}",
    }
    return "".join(replacements.get(c, c) for c in str(value))


def format_p(value):
    if pd.isna(value):
        return "---"
    if value < 0.001:
        return r"$<0.001$"
    return f"{value:.3f}"


def export_latex_table(
    dataframe,
    filename,
    caption,
    label,
    effect_header,
    note,
    float_environment="table*",
):
    """
    Export a six-column detailed result table or overall summary.
    table* spans both columns in a two-column ACM article.
    """
    assert dataframe.shape[1] == 6

    headers = [
        "Measure",
        r"\shortstack{Consortium\\($n=13$)}",
        r"\shortstack{Industry\\($n=11$)}",
        effect_header,
        r"$p$",
        r"$p_{\mathrm{Holm}}$",
    ]

    lines = [
        rf"\begin{{{float_environment}}}[tbp]",
        rf"\caption{{{caption}}}",
        rf"\label{{{label}}}",
        r"\centering",
        r"\small",
        r"\setlength{\tabcolsep}{4pt}",
        r"\renewcommand{\arraystretch}{1.15}",
        (
            r"\begin{tabularx}{\linewidth}{@{}"
            r">{\raggedright\arraybackslash}X"
            r"*{3}{>{\centering\arraybackslash}X}"
            r"cc@{}}"
        ),
        r"\toprule",
        " & ".join(headers) + r" \\",
        r"\midrule",
    ]

    for _, row in dataframe.iterrows():
        values = list(row)

        # Separate the overall comparison from the detailed item rows.
        if str(values[0]).startswith("Overall"):
            lines.append(r"\midrule")

        cells = [
            latex_escape(value)
            for value in values[:4]
        ]
        cells.extend(format_p(value) for value in values[4:])

        lines.append(" & ".join(cells) + r" \\")

    lines.extend([
        r"\bottomrule",
        r"\end{tabularx}",
        r"\vspace{2pt}",
        r"\begin{minipage}{\linewidth}",
        r"\footnotesize\textit{Note.} " + note,
        r"\end{minipage}",
        rf"\end{{{float_environment}}}",
    ])

    latex = "\n".join(lines) + "\n"
    destination = EXPORT_DIR / filename
    destination.write_text(latex, encoding="utf-8")
    print(f"Exported: {destination}")


export_latex_table(
    table_overall,
    filename="agreement_overall_summary.tex",
    caption="Overall Expert Summaries and Group/Session Comparisons",
    label="tab:expert_overall_summary",
    effect_header=r"\shortstack{Pooled $\alpha$\\{[95\% CI]}}",
    note=(
        r"Feasibility entries are percentages rating the single question "
        r"very feasible (4); all remaining responses were feasible (3). "
        r"No agreement coefficient is reported for this single rated object. "
        r"Importance and barrier entries are Krippendorff's $\alpha$ "
        r"[95\% expert-bootstrap CI], using ordinal measurement across "
        r"the three shared technologies and nominal measurement across "
        r"the 11 barriers, respectively. Alpha coefficients are not percentages. "
        r"The overall $p$-values compare group/session response distributions, "
        r"not alpha coefficients: two-sided Fisher's exact test for "
        r"feasibility and whole-expert permutation tests for importance "
        r"and barriers. Holm adjustment covers these three comparisons."
    ),
)


export_latex_table(
    table_feasibility,
    filename="agreement_feasibility.tex",
    caption="Feasibility Ratings and Group/Session Comparison",
    label="tab:expert_feasibility",
    effect_header=r"\shortstack{Difference, pp\\{[95\% CI]}}",
    note=(
        r"Group entries show percentages. Differences are "
        r"industry minus consortium, expressed in percentage points (pp). "
        r"Intervals are 95\% expert-bootstrap confidence intervals. "
        r"One two-sided Fisher's exact test compares the complete "
        r"response distribution; category rows are complementary, "
        r"not separate tests. Holm adjustment covers the three "
        r"overall question-level comparisons."
    ),
)

export_latex_table(
    table_importance,
    filename="agreement_importance.tex",
    caption="Agreement on Shared Importance Ratings and Group/Session Comparison",
    label="tab:expert_importance",
    effect_header=r"\shortstack{Pooled $\alpha$ / $\delta$\\{[95\% CI]}}",
    note=(
        r"The agreement row reports ordinal Krippendorff's $\alpha$ "
        r"with 95\% expert-bootstrap confidence intervals. Solution rows "
        r"show median [Q1, Q3] and Cliff's $\delta$; positive $\delta$ "
        r"indicates higher industry ratings. Analysis includes the three "
        r"solutions shared by both sessions. The overall permutation "
        r"test preserves each expert's complete response profile. "
        r"Holm adjustment covers the three overall question-level comparisons."
    ),
)

export_latex_table(
    table_barriers,
    filename="agreement_barriers.tex",
    caption="Agreement on Barrier Selections and Group/Session Comparison",
    label="tab:expert_barriers",
    effect_header=r"\shortstack{Pooled $\alpha$ /\\difference, pp [95\% CI]}",
    note=(
        r"The agreement row reports nominal Krippendorff's $\alpha$ "
        r"across the 11 barriers. Barrier rows show percentages "
        r"of experts selecting each barrier, with industry-minus-consortium "
        r"differences in percentage points (pp). Intervals are 95\% "
        r"expert-bootstrap confidence intervals. The overall permutation "
        r"test preserves each expert's complete selection profile. "
        r"Holm adjustment covers the three overall question-level comparisons."
    ),
)

Exported: /home/ngirwa/TiiS/second-publication/2-current-analysis/rq1-rerun/experts-consistency/latex-tables/agreement_overall_summary.tex
Exported: /home/ngirwa/TiiS/second-publication/2-current-analysis/rq1-rerun/experts-consistency/latex-tables/agreement_feasibility.tex
Exported: /home/ngirwa/TiiS/second-publication/2-current-analysis/rq1-rerun/experts-consistency/latex-tables/agreement_importance.tex
Exported: /home/ngirwa/TiiS/second-publication/2-current-analysis/rq1-rerun/experts-consistency/latex-tables/agreement_barriers.tex
